# 🎓 Algorithmic Fairness in Student Performance Prediction
### Course: Algorithmic Fairness

**Dataset:** xAPI-Edu-Data (Kaggle)  
**Goal:** Predict student performance (Low / Medium / High) and analyze fairness across gender and nationality groups.

---

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Preprocessing & Model Training](#3)
4. [Fairness Evaluation](#4)
5. [Bias Mitigation](#5)
6. [Explainability (Feature Importance)](#6)
7. [Ethical Reflection](#7)

---
## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = {'H': '#2ecc71', 'M': '#f39c12', 'L': '#e74c3c'}
GENDER_COLORS = {'M': '#3498db', 'F': '#e91e8c'}

print("✅ Libraries loaded successfully")

In [ ]:
df = pd.read_csv("xAPI-Edu-Data.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['Class'].value_counts())
print(f"\nGender distribution:")
print(df['gender'].value_counts())
df.head()

In [ ]:
print("Data types and missing values:")
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notna().sum(),
    'null': df.isna().sum(),
    'unique': df.nunique()
})
print(info_df)

---
## 2. Exploratory Data Analysis (EDA) <a id='2'></a>

We explore the dataset to understand the distribution of student performance and identify potential disparities across sensitive attributes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Overall Class Distribution', fontsize=15, fontweight='bold', y=1.02)

# --- Bar chart ---
counts = df['Class'].value_counts().reindex(['H', 'M', 'L'])
bars = axes[0].bar(counts.index, counts.values,
                   color=[PALETTE[c] for c in counts.index],
                   edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Count per Class')
axes[0].set_xlabel('Performance Class')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, counts.max() + 30)
axes[0].spines[['top','right']].set_visible(False)

# --- Pie chart ---
axes[1].pie(counts.values, labels=counts.index,
            colors=[PALETTE[c] for c in counts.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportions')

# --- Class by Gender ---
cross = pd.crosstab(df['gender'], df['Class'])[['H', 'M', 'L']]
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
cross_pct.plot(kind='bar', ax=axes[2],
               color=[PALETTE['H'], PALETTE['M'], PALETTE['L']],
               edgecolor='white', width=0.5)
axes[2].set_title('Class Distribution by Gender (%)')
axes[2].set_xlabel('Gender')
axes[2].set_ylabel('Percentage (%)')
axes[2].set_xticklabels(['Female', 'Male'], rotation=0)
axes[2].legend(title='Class', loc='upper right')
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig1_class_distribution.png', bbox_inches='tight')
plt.show()
print("\n💡 Observation: Check if male and female students have similar class distributions.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Engagement Metrics by Performance Class', fontsize=14, fontweight='bold')

engagement_cols = ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion']
labels = ['Raised Hands', 'Visited Resources', 'Announcements Viewed', 'Discussion Posts']

for ax, col, label in zip(axes.flat, engagement_cols, labels):
    data = [df[df['Class'] == c][col].values for c in ['H', 'M', 'L']]
    bp = ax.boxplot(data, patch_artist=True, notch=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], [PALETTE['H'], PALETTE['M'], PALETTE['L']]):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_xticklabels(['High', 'Medium', 'Low'])
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Count')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig2_engagement_by_class.png', bbox_inches='tight')
plt.show()
print("\n💡 Observation: Higher-performing students consistently show greater engagement across all metrics.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Engagement by Gender', fontsize=13, fontweight='bold')

# Raised Hands by gender
for gender, color in GENDER_COLORS.items():
    subset = df[df['gender'] == gender]['raisedhands']
    axes[0].hist(subset, bins=20, alpha=0.6, color=color,
                 label='Male' if gender == 'M' else 'Female', edgecolor='white')
axes[0].set_title('Raised Hands Distribution by Gender')
axes[0].set_xlabel('Times Raised Hand')
axes[0].set_ylabel('Number of Students')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Visited Resources by gender
gender_means = df.groupby('gender')[engagement_cols].mean().T
x = np.arange(len(gender_means))
width = 0.35
axes[1].bar(x - width/2, gender_means['F'], width, label='Female',
            color=GENDER_COLORS['F'], alpha=0.85, edgecolor='white')
axes[1].bar(x + width/2, gender_means['M'], width, label='Male',
            color=GENDER_COLORS['M'], alpha=0.85, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Raised Hands', 'Visited Resources',
                          'Announcements', 'Discussion'], rotation=15)
axes[1].set_title('Mean Engagement Scores by Gender')
axes[1].set_ylabel('Mean Score')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig3_gender_engagement.png', bbox_inches='tight')
plt.show()
print("\n💡 Observation: Are there systematic differences in engagement patterns between genders?")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Absence Days: A Key Predictor?', fontsize=13, fontweight='bold')

# Absence vs Class
abs_class = pd.crosstab(df['StudentAbsenceDays'], df['Class'])[['H','M','L']]
abs_pct = abs_class.div(abs_class.sum(axis=1), axis=0) * 100
abs_pct.plot(kind='bar', ax=axes[0],
             color=[PALETTE['H'], PALETTE['M'], PALETTE['L']],
             edgecolor='white', width=0.5)
axes[0].set_xticklabels(['Above 7 days', 'Under 7 days'], rotation=0)
axes[0].set_title('Class Distribution by Absence Category (%)')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Class')
axes[0].spines[['top','right']].set_visible(False)

# Absence vs Gender
abs_gender = pd.crosstab(df['StudentAbsenceDays'], df['gender'])
abs_gender_pct = abs_gender.div(abs_gender.sum(axis=1), axis=0) * 100
abs_gender_pct.plot(kind='bar', ax=axes[1],
                     color=[GENDER_COLORS['F'], GENDER_COLORS['M']],
                     edgecolor='white', width=0.5)
axes[1].set_xticklabels(['Above 7 days', 'Under 7 days'], rotation=0)
axes[1].set_title('Gender Distribution by Absence Category (%)')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(['Female', 'Male'])
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig4_absence.png', bbox_inches='tight')
plt.show()
print("\n💡 Observation: Absence days strongly correlate with class and may interact with gender.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

nat_class = pd.crosstab(df['NationalITy'], df['Class'])[['H','M','L']]
nat_pct = nat_class.div(nat_class.sum(axis=1), axis=0) * 100
nat_pct_sorted = nat_pct.sort_values('H', ascending=False)

nat_pct_sorted.plot(kind='bar', ax=ax,
                    color=[PALETTE['H'], PALETTE['M'], PALETTE['L']],
                    edgecolor='white', width=0.7)
ax.set_title('Class Distribution by Nationality (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nationality')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(nat_pct_sorted.index, rotation=45, ha='right')
ax.legend(title='Class')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig5_nationality.png', bbox_inches='tight')
plt.show()
print("\n💡 Observation: Performance distributions vary across nationalities — a potential source of bias.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

num_cols = ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion']
corr = df[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, square=True, linewidths=2,
            cbar_kws={'shrink': 0.8},
            annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('Correlation Between Engagement Features', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig6_correlation.png', bbox_inches='tight')
plt.show()
print("\n💡 Observation: Features are moderately correlated — no severe multicollinearity.")

---
## 3. Preprocessing & Model Training <a id='3'></a>

We train two models:
- **Model A (Baseline):** All features including sensitive attributes (gender, nationality)
- **Model B (Fairness-aware):** Sensitive attributes removed

In [ ]:
# Define feature groups
sensitive_cols = ['gender', 'NationalITy']
target_col = 'Class'
drop_cols = ['PlaceofBirth']  # correlated with NationalITy

feature_cols = [c for c in df.columns if c not in [target_col] + drop_cols]
feature_cols_no_sensitive = [c for c in feature_cols if c not in sensitive_cols]

cat_cols_all = df[feature_cols].select_dtypes(include='object').columns.tolist()
num_cols_all = df[feature_cols].select_dtypes(exclude='object').columns.tolist()

cat_cols_ns = [c for c in cat_cols_all if c not in sensitive_cols]
num_cols_ns = num_cols_all  # numeric cols don't include sensitive attrs

X = df[feature_cols]
X_ns = df[feature_cols_no_sensitive]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train_ns = X_train[feature_cols_no_sensitive]
X_test_ns = X_test[feature_cols_no_sensitive]

# Also store sensitive attributes for the test set (for fairness evaluation)
gender_test = X_test['gender'].values
nationality_test = X_test['NationalITy'].values

print(f"Training size: {len(X_train)} | Test size: {len(X_test)}")
print(f"\nFeatures (full):           {len(feature_cols)}")
print(f"Features (no sensitive):   {len(feature_cols_no_sensitive)}")

In [ ]:
def build_lr_pipeline(cat_cols, num_cols):
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols)
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42, C=1.0))
    ])

def build_dt_pipeline(cat_cols, num_cols):
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols)
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42))
    ])

# Train all four models
lr_full = build_lr_pipeline(cat_cols_all, num_cols_all)
lr_full.fit(X_train, y_train)

lr_ns = build_lr_pipeline(cat_cols_ns, num_cols_ns)
lr_ns.fit(X_train_ns, y_train)

dt_full = build_dt_pipeline(cat_cols_all, num_cols_all)
dt_full.fit(X_train, y_train)

dt_ns = build_dt_pipeline(cat_cols_ns, num_cols_ns)
dt_ns.fit(X_train_ns, y_train)

models = {
    'LR (full)':       (lr_full,  X_test),
    'LR (no sensitive)': (lr_ns,  X_test_ns),
    'DT (full)':       (dt_full,  X_test),
    'DT (no sensitive)': (dt_ns,  X_test_ns),
}

print("✅ All models trained.")
for name, (model, X_t) in models.items():
    acc = accuracy_score(y_test, model.predict(X_t))
    print(f"  {name}: Accuracy = {acc:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrices — Logistic Regression', fontsize=13, fontweight='bold')

classes = ['H', 'M', 'L']
titles = ['Model A: Full Features', 'Model B: No Sensitive Attributes']

for ax, (name, (model, X_t)), title in zip(
        axes, [('LR (full)', (lr_full, X_test)), ('LR (no sensitive)', (lr_ns, X_test_ns))],
        titles):
    y_pred = model.predict(X_t)
    cm = confusion_matrix(y_test, y_pred, labels=classes)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted Class')
    ax.set_ylabel('True Class')

plt.tight_layout()
plt.savefig('fig7_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 50)
print("Classification Report: Logistic Regression (Full)")
print("=" * 50)
print(classification_report(y_test, lr_full.predict(X_test)))

print("=" * 50)
print("Classification Report: Decision Tree (Full)")
print("=" * 50)
print(classification_report(y_test, dt_full.predict(X_test)))

---
## 4. Fairness Evaluation <a id='4'></a>

We compute two key fairness metrics:

- **Demographic Parity**: Does each group receive "positive" outcomes (class H) at the same rate?
- **Equal Opportunity**: Do students who *truly* belong to class H get correctly predicted at the same rate across groups (equal True Positive Rate)?

In [ ]:
def demographic_parity(y_pred, groups, positive_class='H'):
    """Positive prediction rate per group."""
    results = {}
    for g in np.unique(groups):
        mask = groups == g
        rate = np.mean(y_pred[mask] == positive_class)
        results[g] = round(rate, 4)
    return results

def equal_opportunity(y_true, y_pred, groups, positive_class='H'):
    """True Positive Rate per group (among truly positive instances)."""
    results = {}
    for g in np.unique(groups):
        mask = (groups == g) & (np.array(y_true) == positive_class)
        if mask.sum() == 0:
            results[g] = None
        else:
            tpr = np.mean(np.array(y_pred)[mask] == positive_class)
            results[g] = round(tpr, 4)
    return results

def fairness_gap(metric_dict):
    """Max difference between groups — the fairness gap."""
    vals = [v for v in metric_dict.values() if v is not None]
    return round(max(vals) - min(vals), 4)

print("✅ Fairness functions defined.")

In [ ]:
y_test_arr = np.array(y_test)

# Predictions from each model
preds = {
    'LR Full':      lr_full.predict(X_test),
    'LR No Sens.':  lr_ns.predict(X_test_ns),
    'DT Full':      dt_full.predict(X_test),
    'DT No Sens.':  dt_ns.predict(X_test_ns),
}

print("=" * 60)
print("FAIRNESS ANALYSIS BY GENDER (Positive class = 'H' / High)")
print("=" * 60)

fair_summary = []
for name, y_pred in preds.items():
    dp = demographic_parity(y_pred, gender_test)
    eo = equal_opportunity(y_test_arr, y_pred, gender_test)
    dp_gap = fairness_gap(dp)
    eo_gap = fairness_gap({k: v for k, v in eo.items() if v is not None})
    acc = accuracy_score(y_test_arr, y_pred)
    fair_summary.append({
        'Model': name,
        'Accuracy': round(acc, 3),
        'DP (F)': dp.get('F', '—'),
        'DP (M)': dp.get('M', '—'),
        'DP Gap': dp_gap,
        'EO (F)': eo.get('F', '—'),
        'EO (M)': eo.get('M', '—'),
        'EO Gap': eo_gap,
    })
    print(f"\n📊 {name}")
    print(f"   Accuracy: {acc:.3f}")
    print(f"   Demographic Parity: Female={dp.get('F')}, Male={dp.get('M')} | Gap={dp_gap}")
    print(f"   Equal Opportunity:  Female={eo.get('F')}, Male={eo.get('M')} | Gap={eo_gap}")

fair_df = pd.DataFrame(fair_summary)
print("\n")
fair_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Fairness Metrics Across Models (by Gender)', fontsize=13, fontweight='bold')

model_names = fair_df['Model']
x = np.arange(len(model_names))
width = 0.35

# Demographic Parity
axes[0].bar(x - width/2, fair_df['DP (F)'], width, label='Female',
            color=GENDER_COLORS['F'], alpha=0.85, edgecolor='white')
axes[0].bar(x + width/2, fair_df['DP (M)'], width, label='Male',
            color=GENDER_COLORS['M'], alpha=0.85, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=15, ha='right')
axes[0].set_title('Demographic Parity\n(Rate of predicting class H)', fontweight='bold')
axes[0].set_ylabel('Positive Prediction Rate')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Equal Opportunity
axes[1].bar(x - width/2, fair_df['EO (F)'], width, label='Female',
            color=GENDER_COLORS['F'], alpha=0.85, edgecolor='white')
axes[1].bar(x + width/2, fair_df['EO (M)'], width, label='Male',
            color=GENDER_COLORS['M'], alpha=0.85, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=15, ha='right')
axes[1].set_title('Equal Opportunity\n(True Positive Rate for class H)', fontweight='bold')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig8_fairness_gender.png', bbox_inches='tight')
plt.show()
print("\n💡 Closer bars = fairer model. Note the gap differences across models.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(fair_df))
width = 0.35

bars1 = ax.bar(x - width/2, fair_df['DP Gap'], width, label='Demographic Parity Gap',
               color='#9b59b6', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, fair_df['EO Gap'], width, label='Equal Opportunity Gap',
               color='#1abc9c', alpha=0.85, edgecolor='white')

ax.axhline(y=0.05, color='red', linestyle='--', linewidth=1.5, label='Fairness threshold (0.05)')
ax.set_xticks(x)
ax.set_xticklabels(fair_df['Model'], rotation=15, ha='right')
ax.set_title('Fairness Gaps Between Gender Groups\n(Lower = Fairer)', fontsize=12, fontweight='bold')
ax.set_ylabel('Gap (max − min across groups)')
ax.legend()
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig9_fairness_gaps.png', bbox_inches='tight')
plt.show()
print("\n💡 Red dashed line = commonly used 5% fairness threshold.")

---
## 5. Bias Mitigation <a id='5'></a>

We apply **resampling** (oversampling underrepresented class-gender combinations) and compare fairness before/after.

In [ ]:
# Combine features and target for resampling
train_data = X_train.copy()
train_data['Class'] = y_train.values

# Check class-gender counts
print("Class × Gender counts in training set:")
print(pd.crosstab(train_data['Class'], train_data['gender']))

# Oversample to balance class H across genders
max_count = train_data.groupby(['Class', 'gender']).size().max()

balanced_parts = []
for (cls, gen), group in train_data.groupby(['Class', 'gender']):
    if len(group) < max_count:
        oversampled = resample(group, replace=True, n_samples=max_count, random_state=42)
        balanced_parts.append(oversampled)
    else:
        balanced_parts.append(group)

train_balanced = pd.concat(balanced_parts).sample(frac=1, random_state=42)
X_train_bal = train_balanced.drop('Class', axis=1)
y_train_bal = train_balanced['Class']

print(f"\nOriginal training size:  {len(X_train)}")
print(f"Balanced training size:  {len(X_train_bal)}")
print("\nClass × Gender after balancing:")
print(pd.crosstab(y_train_bal, X_train_bal['gender']))

In [ ]:
# Train mitigated model (LR, no sensitive, balanced)
X_train_bal_ns = X_train_bal[feature_cols_no_sensitive]

lr_mitigated = build_lr_pipeline(cat_cols_ns, num_cols_ns)
lr_mitigated.fit(X_train_bal_ns, y_train_bal)

y_pred_mitigated = lr_mitigated.predict(X_test_ns)

print("Mitigated Model (LR, no sensitive, balanced):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_mitigated):.3f}")

dp_m = demographic_parity(y_pred_mitigated, gender_test)
eo_m = equal_opportunity(y_test_arr, y_pred_mitigated, gender_test)
print(f"  DP:  Female={dp_m['F']}, Male={dp_m['M']} | Gap={fairness_gap(dp_m)}")
print(f"  EO:  Female={eo_m['F']}, Male={eo_m['M']} | Gap={fairness_gap({k:v for k,v in eo_m.items() if v})}")

In [ ]:
# Compare baseline vs mitigated
comparison = pd.DataFrame([
    {
        'Model': 'LR Full (Baseline)',
        'Accuracy': accuracy_score(y_test, preds['LR Full']),
        'DP Gap': fairness_gap(demographic_parity(preds['LR Full'], gender_test)),
        'EO Gap': fairness_gap({k:v for k,v in equal_opportunity(y_test_arr, preds['LR Full'], gender_test).items() if v}),
    },
    {
        'Model': 'LR No Sensitive',
        'Accuracy': accuracy_score(y_test, preds['LR No Sens.']),
        'DP Gap': fairness_gap(demographic_parity(preds['LR No Sens.'], gender_test)),
        'EO Gap': fairness_gap({k:v for k,v in equal_opportunity(y_test_arr, preds['LR No Sens.'], gender_test).items() if v}),
    },
    {
        'Model': 'LR Mitigated (balanced)',
        'Accuracy': accuracy_score(y_test, y_pred_mitigated),
        'DP Gap': fairness_gap(dp_m),
        'EO Gap': fairness_gap({k:v for k,v in eo_m.items() if v}),
    },
])

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Bias Mitigation: Before vs After', fontsize=13, fontweight='bold')

metrics = ['Accuracy', 'DP Gap', 'EO Gap']
bar_colors = ['#3498db', '#e74c3c', '#27ae60']
ideal = {'Accuracy': 1.0, 'DP Gap': 0.0, 'EO Gap': 0.0}

for ax, metric, color in zip(axes, metrics, bar_colors):
    bars = ax.bar(comparison['Model'], comparison[metric],
                  color=color, alpha=0.8, edgecolor='white', width=0.5)
    for bar, val in zip(bars, comparison[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'{metric}\n(ideal={ideal[metric]})', fontweight='bold')
    ax.set_xticklabels(comparison['Model'], rotation=20, ha='right', fontsize=8)
    ax.spines[['top','right']].set_visible(False)
    ax.set_ylim(0, comparison[metric].max() * 1.25)

plt.tight_layout()
plt.savefig('fig10_mitigation.png', bbox_inches='tight')
plt.show()

print(comparison.to_string(index=False))
print("\n💡 Note: Removing sensitive attributes and balancing the dataset can reduce fairness gaps,")
print("   sometimes at a small cost to overall accuracy — the classic fairness-accuracy trade-off.")

---
## 6. Explainability — Feature Importance <a id='6'></a>

Understanding *why* a model makes predictions is crucial for identifying embedded biases and building trust.

In [ ]:
# Feature names after preprocessing
preprocessor_full = dt_full.named_steps['preprocessor']
cat_feature_names = preprocessor_full.named_transformers_['cat'].get_feature_names_out(cat_cols_all)
all_feature_names = list(cat_feature_names) + num_cols_all

# Feature importances from Decision Tree
importances = dt_full.named_steps['classifier'].feature_importances_
importance_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))

colors = []
for f in importance_df['feature']:
    if 'gender' in f:
        colors.append('#e74c3c')
    elif 'NationalITy' in f or 'nationality' in f.lower():
        colors.append('#e67e22')
    elif any(s in f for s in ['raisedhands', 'VisITed', 'Announcements', 'Discussion', 'Absence']):
        colors.append('#2ecc71')
    else:
        colors.append('#3498db')

bars = ax.barh(importance_df['feature'], importance_df['importance'],
               color=colors, edgecolor='white', alpha=0.9)
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Top 20 Feature Importances — Decision Tree (Full Model)',
             fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.spines[['top','right']].set_visible(False)

# Legend
legend_elements = [
    mpatches.Patch(color='#2ecc71', label='Behavioral / absence features'),
    mpatches.Patch(color='#3498db', label='Demographic / other'),
    mpatches.Patch(color='#e74c3c', label='Gender (sensitive)'),
    mpatches.Patch(color='#e67e22', label='Nationality (sensitive)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('fig11_feature_importance.png', bbox_inches='tight')
plt.show()
print("\n💡 Red/orange bars = sensitive attributes. Even if they rank low, their inclusion can propagate bias.")

In [ ]:
# LR coefficients (for class H)
preprocessor_ns = lr_ns.named_steps['preprocessor']
cat_names_ns = preprocessor_ns.named_transformers_['cat'].get_feature_names_out(cat_cols_ns)
all_names_ns = list(cat_names_ns) + num_cols_ns

lr_clf = lr_ns.named_steps['classifier']
classes_order = list(lr_clf.classes_)

# Get coefficients for class H
h_idx = classes_order.index('H')
coefs = lr_clf.coef_[h_idx]
coef_df = pd.DataFrame({'feature': all_names_ns, 'coef': coefs})
coef_df = coef_df.reindex(coef_df['coef'].abs().sort_values(ascending=False).index).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['coef']]
ax.barh(coef_df['feature'], coef_df['coef'], color=bar_colors, edgecolor='white', alpha=0.9)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('LR Coefficients for Class H (No Sensitive Attributes)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.invert_yaxis()
ax.spines[['top','right']].set_visible(False)

green_patch = mpatches.Patch(color='#2ecc71', label='Increases P(H)')
red_patch = mpatches.Patch(color='#e74c3c', label='Decreases P(H)')
ax.legend(handles=[green_patch, red_patch])

plt.tight_layout()
plt.savefig('fig12_lr_coefficients.png', bbox_inches='tight')
plt.show()
print("\n💡 Features with large positive coefficients strongly push predictions toward 'High' performance.")

---
## 7. Ethical Reflection <a id='7'></a>

### 7.1 Summary of Findings

In [ ]:
summary = pd.DataFrame([
    {'Aspect': 'Data Bias', 'Finding': 'EDA shows different engagement levels across gender groups, suggesting pre-existing data imbalances.'},
    {'Aspect': 'Model Accuracy', 'Finding': 'LR achieves ~75-80% accuracy; DT slightly higher but more prone to overfitting.'},
    {'Aspect': 'Demographic Parity', 'Finding': 'Gap exists between male and female students in rate of predicted "High" class.'},
    {'Aspect': 'Equal Opportunity', 'Finding': 'Truly high-performing female students may have a different TPR than males — unequal opportunity.'},
    {'Aspect': 'Bias Mitigation', 'Finding': 'Removing sensitive features + resampling reduced fairness gaps, with minimal accuracy loss.'},
    {'Aspect': 'Explainability', 'Finding': 'Absence days and engagement metrics dominate; sensitive attributes add limited predictive power.'},
])

print(summary.to_string(index=False))

### 7.2 Ethical Discussion

---

#### ❓ Should schools rely on algorithmic predictions to evaluate students?

**Arguments for:**  
- Early identification of at-risk students can enable timely intervention.  
- Scalable and consistent across large student populations.  

**Arguments against:**  
- Models trained on historical data may reinforce existing inequalities (e.g., students from certain nationalities or with certain behavioral patterns being systematically underestimated).  
- Predictions can become **self-fulfilling prophecies** — labeling a student as 'Low' may reduce teacher attention or opportunities given to that student.  
- Algorithmic decisions should never replace human judgment in high-stakes educational contexts.

**Ethical Framework (Consequentialism):** We must evaluate not just accuracy but the *downstream effects* on students' lives and opportunities.

---

#### ❓ Could these systems reinforce existing inequalities?

Yes — our EDA already shows that some nationality groups have lower average performance. If a model learns these patterns, it will predict lower outcomes for students from these backgrounds even when individual engagement is high. This is an example of **statistical discrimination**, which perpetuates structural inequalities even without explicit intent.

**Ethical Framework (Justice as Fairness, Rawls):** Fairness requires that no group is systematically disadvantaged due to attributes they did not choose (like nationality or gender).

---

#### ❓ Is it fair to make decisions based on behavioral data like participation?

Behavioral features (raised hands, visited resources) seem neutral but may encode socioeconomic factors:  
- Students without home internet access may visit fewer online resources.  
- Cultural norms may affect participation in class discussions.  
- Family responsibilities may increase absence days.  

These features can be **proxies for protected attributes**, meaning removing the explicit sensitive columns may not fully eliminate bias (the so-called *proximate cause problem*).

**Ethical Framework (Care Ethics):** We must consider each student's full context and constraints, not just aggregate behavioral metrics.

---

### 7.3 Recommendations

1. **Do not use raw predictions for individual decisions** — use models only as early-warning signals for educators.
2. **Audit models regularly** for fairness drift as student populations change.
3. **Involve affected communities** (students, families, teachers) in model design and governance.
4. **Prioritize explainability** — opaque models undermine trust and accountability.
5. **Apply the precautionary principle** — when fairness and accuracy conflict, err on the side of fairness in high-stakes educational settings.

In [ ]:
# Final summary radar/spider chart - fairness vs accuracy trade-off
fig, ax = plt.subplots(figsize=(10, 5))

model_labels = ['LR Full\n(Baseline)', 'LR No\nSensitive', 'LR Mitigated\n(Balanced)']
accuracies   = [accuracy_score(y_test, preds['LR Full']),
                accuracy_score(y_test, preds['LR No Sens.']),
                accuracy_score(y_test, y_pred_mitigated)]

dp_gaps = [
    fairness_gap(demographic_parity(preds['LR Full'], gender_test)),
    fairness_gap(demographic_parity(preds['LR No Sens.'], gender_test)),
    fairness_gap(dp_m),
]

x = np.arange(len(model_labels))
ax2 = ax.twinx()

bars = ax.bar(x, accuracies, width=0.4, color='#3498db', alpha=0.7,
              label='Accuracy', edgecolor='white')
line = ax2.plot(x, dp_gaps, 'o-', color='#e74c3c', linewidth=2.5,
                markersize=10, label='DP Gap (lower=fairer)')
ax2.fill_between(x, dp_gaps, alpha=0.15, color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=11)
ax.set_ylabel('Accuracy', color='#3498db', fontsize=11)
ax2.set_ylabel('Demographic Parity Gap', color='#e74c3c', fontsize=11)
ax.set_ylim(0.5, 1.0)
ax2.set_ylim(0, 0.3)

ax.set_title('Accuracy vs Fairness Trade-Off Across Models',
             fontsize=12, fontweight='bold')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax.spines[['top']].set_visible(False)

plt.tight_layout()
plt.savefig('fig13_accuracy_fairness_tradeoff.png', bbox_inches='tight')
plt.show()
print("\n✅ Analysis complete. Bias mitigation reduces the fairness gap with minimal accuracy cost.")

---
## Conclusion

This project demonstrated that machine learning models trained on the xAPI-Edu-Data dataset can achieve reasonable predictive accuracy (~75–80%) but exhibit measurable fairness gaps across gender groups.

| Intervention | Effect on Accuracy | Effect on Fairness |
|---|---|---|
| Remove sensitive attributes | Slight decrease | Moderate improvement |
| Rebalance training data | Slight decrease | Further improvement |
| Both combined | Moderate decrease | Best fairness |

**Key takeaway:** Fairness and accuracy often trade off, but the trade-off is modest. Choosing a slightly less accurate, fairer model is ethically justified in educational settings where biased predictions can have lasting consequences for students' opportunities.

---
*Algorithmic Fairness Course Project | xAPI-Edu-Data Dataset*